# 1 — Setup & Dataset
**Goal:** Verify GPU, install dependencies, mount Drive, clone the repo, extract the dataset, and confirm all paths are correct before any training.

Run this notebook **once** at the start of every new Colab session.

## 1.1 — Verify GPU
Check that a GPU is attached. Go to **Runtime → Change runtime type → T4 GPU** if this fails.

In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device       : {device}')

if torch.cuda.is_available():
    print(f'GPU name     : {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU memory   : {total_mem:.1f} GB')
else:
    print('⚠️  No GPU detected — switch to GPU runtime before training')

Device       : cuda
GPU name     : Tesla T4
GPU memory   : 14.6 GB


## 1.2 — Install Dependencies
Install all required packages. This takes ~1 minute on first run.

In [2]:
!pip install -q torch torchaudio transformers scipy scikit-learn matplotlib soundfile pyyaml
print('✅ Dependencies installed')

✅ Dependencies installed


## 1.3 — Mount Google Drive
Drive holds the dataset zip, trained models, and all saved outputs.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/deepfake_detector'
SAVE_DIR   = f'{DRIVE_ROOT}/models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Drive root : {DRIVE_ROOT}')
print(f'Save dir   : {SAVE_DIR}')
print(f'Drive OK   : {os.path.exists(DRIVE_ROOT)}')

Mounted at /content/drive
Drive root : /content/drive/MyDrive/deepfake_detector
Save dir   : /content/drive/MyDrive/deepfake_detector/models
Drive OK   : True


## 1.4 — Clone GitHub Repo
Pull the project src files directly from GitHub so every notebook can import them.

In [4]:
import os, sys

PROJECT_DIR = '/content/project'

if os.path.exists(PROJECT_DIR):
    print('Repo already cloned — pulling latest...')
    !cd {PROJECT_DIR} && git pull
else:
    REPO_URL = 'https://github.com/Arjun11x/deepfake-audio-detection.git'
    !git clone {REPO_URL} {PROJECT_DIR}

# Add src/ to Python path so notebooks can import config, models, dataset, utils
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'\n✅ Project dir : {PROJECT_DIR}')
print(f'✅ src/ on path : {SRC_DIR}')
print(f'Files in src/  : {os.listdir(SRC_DIR)}')

Cloning into '/content/project'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 0), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 18.41 KiB | 18.41 MiB/s, done.

✅ Project dir : /content/project
✅ src/ on path : /content/project/src
Files in src/  : ['train.py', 'utils.py', 'config.py', 'dataset.py', 'inference.py', 'models.py', '__init__.py', 'evaluate.py']


## 1.5 — Configure Paths
Set `ENV = 'colab'` in config so all paths point to Colab + Drive locations.

In [5]:
import sys, os
import importlib
import config

# Manually override all Colab paths directly
config.ENV         = "colab"
config.DATASET_ROOT = "/content/asvspoof2019/LA"
config.AUDIO_DIRS  = {
    "train" : "/content/asvspoof2019/LA/ASVspoof2019_LA_train/flac",
    "dev"   : "/content/asvspoof2019/LA/ASVspoof2019_LA_dev/flac",
    "eval"  : "/content/asvspoof2019/LA/ASVspoof2019_LA_eval/flac",
}
config.PROTOCOL_FILES = {
    "train" : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
    "dev"   : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt",
    "eval"  : "/content/asvspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
}
config.SAVE_DIR        = "/content/drive/MyDrive/deepfake_detector/models"
config.BEST_MODEL_PATH = f"{config.SAVE_DIR}/student_best.pth"
config.CHECKPOINT_PATH = f"{config.SAVE_DIR}/training_checkpoint.pth"
config.FINAL_MODEL_PATH= f"{config.SAVE_DIR}/student_final.pth"

# Print paths only — no hyperparameters
print(f"{'='*50}")
print(f"  Active ENV    : {config.ENV}")
print(f"  Dataset root  : {config.DATASET_ROOT}")
print(f"  Save dir      : {config.SAVE_DIR}")
print(f"{'='*50}")
print(f"  train audio   : {config.AUDIO_DIRS['train']}")
print(f"  dev audio     : {config.AUDIO_DIRS['dev']}")
print(f"  eval audio    : {config.AUDIO_DIRS['eval']}")
print(f"{'='*50}")

  Active ENV    : colab
  Dataset root  : /content/asvspoof2019/LA
  Save dir      : /content/drive/MyDrive/deepfake_detector/models
  train audio   : /content/asvspoof2019/LA/ASVspoof2019_LA_train/flac
  dev audio     : /content/asvspoof2019/LA/ASVspoof2019_LA_dev/flac
  eval audio    : /content/asvspoof2019/LA/ASVspoof2019_LA_eval/flac


## 1.6 — Extract Dataset
Unzip `LA.zip` from Drive to `/content/asvspoof2019/`. Skipped automatically if already extracted.

In [6]:
import os

ZIP_PATH    = '/content/drive/MyDrive/LA.zip'
EXTRACT_DIR = '/content/asvspoof2019'
LA_DIR      = os.path.join(EXTRACT_DIR, 'LA')

if os.path.exists(LA_DIR):
    print(f'✅ Dataset already extracted at {LA_DIR}')
else:
    print(f'Extracting {ZIP_PATH} ...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
    print(f'✅ Extraction complete')

print(f'\nContents of {LA_DIR}:')
for item in sorted(os.listdir(LA_DIR)):
    print(f'  {item}')

Extracting /content/drive/MyDrive/LA.zip ...
✅ Extraction complete

Contents of /content/asvspoof2019/LA:
  ASVspoof2019_LA_asv_protocols
  ASVspoof2019_LA_asv_scores
  ASVspoof2019_LA_cm_protocols
  ASVspoof2019_LA_dev
  ASVspoof2019_LA_eval
  ASVspoof2019_LA_train
  README.LA.txt


## 1.7 — Verify Dataset Paths
Confirm all audio folders and protocol files exist and contain the expected file counts.

In [8]:
import os

# Expected counts
EXPECTED = {
    'train' : {'flac': 25380, 'protocol': True},
    'dev'   : {'flac': 24986, 'protocol': True},
    'eval'  : {'flac': 71933, 'protocol': True},
}

all_ok = True

for subset in ['train', 'dev', 'eval']:
    audio_dir = config.AUDIO_DIRS[subset]
    proto     = config.PROTOCOL_FILES[subset]

    audio_exists = os.path.exists(audio_dir)
    proto_exists = os.path.exists(proto)
    flac_count   = len([f for f in os.listdir(audio_dir) if f.endswith('.flac')]) if audio_exists else 0
    expected_n   = EXPECTED[subset]['flac']

    status = '✅' if (audio_exists and proto_exists and flac_count == expected_n) else '❌'
    if status == '❌':
        all_ok = False

    print(f'{status} {subset:<6} | audio: {audio_exists} | protocol: {proto_exists} | flac files: {flac_count}/{expected_n}')

print()
if all_ok:
    print('✅ All paths verified — ready to proceed to notebook 2')
else:
    print('❌ Some paths failed — re-check extraction or Drive mounting')

✅ train  | audio: True | protocol: True | flac files: 25380/25380
✅ dev    | audio: True | protocol: True | flac files: 24986/24986
✅ eval   | audio: True | protocol: True | flac files: 71933/71933

✅ All paths verified — ready to proceed to notebook 2


## 1.8 — Verify Protocol File Format
Sanity-check that protocol files parse correctly. Each line should yield a file_id and bonafide/spoof label.

In [9]:
for subset in ['train', 'dev', 'eval']:
    proto = config.PROTOCOL_FILES[subset]
    with open(proto, 'r') as f:
        lines = f.readlines()

    real_count = sum(1 for l in lines if l.strip().split()[4] == 'bonafide')
    fake_count = sum(1 for l in lines if l.strip().split()[4] == 'spoof')

    # Show first 2 lines as format check
    sample_parts = lines[0].strip().split()
    print(f'--- {subset} ---')
    print(f'  Total lines : {len(lines)}')
    print(f'  Real        : {real_count} | Fake: {fake_count}')
    print(f'  Sample line : {lines[0].strip()}')
    print(f'  file_id     : {sample_parts[1]} | label: {sample_parts[4]}')
    print()

--- train ---
  Total lines : 25380
  Real        : 2580 | Fake: 22800
  Sample line : LA_0079 LA_T_1138215 - - bonafide
  file_id     : LA_T_1138215 | label: bonafide

--- dev ---
  Total lines : 24844
  Real        : 2548 | Fake: 22296
  Sample line : LA_0069 LA_D_1047731 - - bonafide
  file_id     : LA_D_1047731 | label: bonafide

--- eval ---
  Total lines : 71237
  Real        : 7355 | Fake: 63882
  Sample line : LA_0039 LA_E_2834763 - A11 spoof
  file_id     : LA_E_2834763 | label: spoof



## 1.9 — Verify src/ Imports
Quick smoke test to confirm all modules import cleanly.

In [10]:
from models  import MobileStudentCNN, Wav2VecTeacher
from dataset import AudioDeepfakeDataset
from utils   import load_asvspoof2019, compute_eer, kd_loss
import config

# Quick shape test — no GPU needed
import torch
dummy = torch.randn(2, 1, 64, 126)
model = MobileStudentCNN()
out   = model(dummy)

assert out.shape == torch.Size([2, 2]), f'Shape mismatch: {out.shape}'
total_params = sum(p.numel() for p in model.parameters())

print(f'✅ MobileStudentCNN output : {out.shape}')
print(f'✅ Parameters              : {total_params:,} (~{total_params/1e6:.2f}M)')
print(f'✅ All imports OK — ready for notebook 2')

✅ MobileStudentCNN output : torch.Size([2, 2])
✅ Parameters              : 230,058 (~0.23M)
✅ All imports OK — ready for notebook 2


---
## ✅ Setup Complete

| Step | Status |
|------|--------|
| GPU verified | ✅ |
| Dependencies installed | ✅ |
| Drive mounted | ✅ |
| Repo cloned + src/ on path | ✅ |
| Dataset extracted | ✅ |
| Paths verified | ✅ |
| Imports working | ✅ |

**Next → `2_EDA.ipynb`** — explore the dataset with spectrograms, waveforms, and class distribution plots.